In [1]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('/home/ililo/Dev/merged.csv')
df['ts'] = pd.to_datetime(df['ts']).dt.tz_convert('Africa/Addis_Ababa')
df

,ts,ms_played,track,artist,album,reason_start,reason_end,skipped,id,popularity,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,2025-03-04 09:17:12+03:00,195626,streatham,dave,psychodrama,clickrow,fwdbtn,True,streatham_dave,67,...,2,-7.670,1,0.0949,0.4960,0.000000,0.1090,0.638,124.015,4
1,2025-03-04 09:17:13+03:00,576,black,dave,psychodrama,fwdbtn,fwdbtn,True,black_dave,59,...,7,-7.150,1,0.4220,0.3590,0.000000,0.1160,0.334,81.584,4
2,2025-03-04 09:25:27+03:00,252149,screwface capital,dave,psychodrama,trackdone,trackdone,False,screwface capital_dave,68,...,0,-7.560,1,0.0486,0.5230,0.000037,0.1980,0.427,112.015,4
3,2025-03-04 09:50:58+03:00,144677,environment,dave,psychodrama,trackdone,fwdbtn,True,environment_dave,58,...,11,-10.133,0,0.4610,0.5430,0.000000,0.4480,0.522,84.268,4
4,2025-03-04 09:51:00+03:00,1194,voices,dave,psychodrama,fwdbtn,fwdbtn,True,voices_dave,59,...,11,-7.570,0,0.0773,0.1810,0.000182,0.1540,0.579,116.942,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22722,2024-12-14 19:47:15+03:00,192600,gasolina,daddy yankee,barrio fino (bonus track version),trackdone,trackdone,False,gasolina_daddy yankee,3,...,0,-5.157,1,0.0561,0.3880,0.000004,0.0692,0.758,96.009,4
22723,2024-12-14 19:47:15+03:00,192600,gasolina,daddy yankee,barrio fino (bonus track version),trackdone,trackdone,False,gasolina_daddy yankee,6,...,0,-6.274,1,0.0626,0.2810,0.000000,0.0806,0.814,96.002,4
22724,2024-12-14 19:47:43+03:00,26216,who you foolin,gunna,drip or drown 2,trackdone,fwdbtn,True,who you foolin_gunna,72,...,0,-7.653,1,0.1880,0.0942,0.000000,0.2300,0.756,115.690,4
22725,2024-12-14 20:13:06+03:00,204174,addicted,jorja smith,be right back,trackdone,trackdone,False,addicted_jorja smith,74,...,2,-10.098,1,0.0519,0.2410,0.010700,0.1600,0.153,134.023,4


In [ ]:
def create_time_features(df):
    df = df.copy()

    df['hour'] = df['ts'].dt.hour
    df['day_of_week'] = df['ts'].dt.dayofweek
    df['month'] = df['ts'].dt.month
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(bool)

    df['time_of_day'] = pd.cut(df['hour'], 
                            bins=[0, 4, 12, 17, 24],
                            labels=['Mid-Night', 'Morning', 'Afternoon', 'Night'],
                            include_lowest=True)
    return df

In [4]:
def create_completion_features(df):
    df = df.copy()
    df['rewound'] = (df['ms_played'] > df['duration_ms']).astype(bool)
    df['completion_rate'] = (df['ms_played']/df['duration_ms'])*100

    return df

In [5]:
def create_categorical_features(df):
    df=df.copy()
    mapping = {
        'clickrow': 'new',
        'fwdbtn': 'user',
        'trackdone': 'natural',
        'backbtn': 'user',
        'endplay': 'nature'}

    df['reason_start'] = df['reason_start'].map(mapping)
    df['reason_end'] = df['reason_end'].map(mapping)

    return df

In [6]:
def create_engagement_features(df):

    df = df.copy()
    
    track_stats= df.groupby('id').agg(
        avg_completion_rate=('completion_rate', 'mean'),
        streams = ('id', 'count')
    ).reset_index()

    df = pd.merge(df, track_stats, on='id', how='inner')
    return df

In [7]:
def apply_features(train_df, test_df=None):

    train_df = create_time_features(train_df)
    train_df = create_categorical_features(train_df)
    train_df = create_completion_features(train_df)
    train_df = create_engagement_features(train_df)

    if test_df is not None:
        test_df = create_time_features(test_df)
        test_df = create_categorical_features(test_df)
        test_df = create_completion_features(test_df)
        test_df = create_engagement_features(test_df)

        track_stats = train_df[['id', 'avg_completion_rate', 'streams']]
        test_df.drop(['avg_completion_rate', 'streams'], axis=1)
        test_df = pd.merge(test_df, track_stats, how='left')

        global_avg = train_df['completion_rate'].mean()
        test_df['completion_rate'] = test_df['completion_rate'].fillna(global_avg)
        test_df['streams'] = test_df['streams'].fillna(0)

    return train_df, test_df

In [8]:
train_df = apply_features(df)
train_df

(                             ts  ms_played              track        artist  \
 0     2025-03-04 09:17:12+03:00     195626          streatham          dave   
 1     2025-03-04 09:17:13+03:00        576              black          dave   
 2     2025-03-04 09:25:27+03:00     252149  screwface capital          dave   
 3     2025-03-04 09:50:58+03:00     144677        environment          dave   
 4     2025-03-04 09:51:00+03:00       1194             voices          dave   
 ...                         ...        ...                ...           ...   
 22722 2024-12-14 19:47:15+03:00     192600           gasolina  daddy yankee   
 22723 2024-12-14 19:47:15+03:00     192600           gasolina  daddy yankee   
 22724 2024-12-14 19:47:43+03:00      26216     who you foolin         gunna   
 22725 2024-12-14 20:13:06+03:00     204174           addicted   jorja smith   
 22726 2024-12-14 20:17:53+03:00     236120                fml    kanye west   
 
                                    al

In [10]:
new = create_completion_features(df)
new = create_engagement_features(new)
new
#top_artists = new['id'].value_counts().head(28)
#top_artists


,ts,ms_played,track,artist,album,reason_start,reason_end,skipped,id,popularity,...,acousticness,instrumentalness,liveness,valence,tempo,time_signature,rewound,completion_rate,avg_completion_rate,streams
0,2025-03-04 06:17:12+00:00,195626,streatham,dave,psychodrama,clickrow,fwdbtn,True,streatham_dave,67,...,0.4960,0.000000,0.1090,0.638,124.015,4,False,95.093331,74.013807,64
1,2025-03-04 06:17:13+00:00,576,black,dave,psychodrama,fwdbtn,fwdbtn,True,black_dave,59,...,0.3590,0.000000,0.1160,0.334,81.584,4,False,0.252322,53.824105,48
2,2025-03-04 06:25:27+00:00,252149,screwface capital,dave,psychodrama,trackdone,trackdone,False,screwface capital_dave,68,...,0.5230,0.000037,0.1980,0.427,112.015,4,False,99.627015,67.971097,79
3,2025-03-04 06:50:58+00:00,144677,environment,dave,psychodrama,trackdone,fwdbtn,True,environment_dave,58,...,0.5430,0.000000,0.4480,0.522,84.268,4,False,71.363320,61.064245,69
4,2025-03-04 06:51:00+00:00,1194,voices,dave,psychodrama,fwdbtn,fwdbtn,True,voices_dave,59,...,0.1810,0.000182,0.1540,0.579,116.942,4,False,0.601975,30.730084,25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22722,2024-12-14 16:47:15+00:00,192600,gasolina,daddy yankee,barrio fino (bonus track version),trackdone,trackdone,False,gasolina_daddy yankee,3,...,0.3880,0.000004,0.0692,0.758,96.009,4,True,100.013501,85.455474,252
22723,2024-12-14 16:47:15+00:00,192600,gasolina,daddy yankee,barrio fino (bonus track version),trackdone,trackdone,False,gasolina_daddy yankee,6,...,0.2810,0.000000,0.0806,0.814,96.002,4,True,111.415134,85.455474,252
22724,2024-12-14 16:47:43+00:00,26216,who you foolin,gunna,drip or drown 2,trackdone,fwdbtn,True,who you foolin_gunna,72,...,0.0942,0.000000,0.2300,0.756,115.690,4,False,17.299837,84.226097,13
22725,2024-12-14 17:13:06+00:00,204174,addicted,jorja smith,be right back,trackdone,trackdone,False,addicted_jorja smith,74,...,0.2410,0.010700,0.1600,0.153,134.023,4,False,99.999510,54.356518,50
